# 04 - Stable Diffusion 完整功能演示

使用 Hugging Face diffusers 库体验完整的 Stable Diffusion 功能。

**前置条件**: DSW 服务器，A10 GPU，设置 HuggingFace 镜像：
```bash
export HF_ENDPOINT=https://hf-mirror.com
```

In [ ]:
import torch
from diffusers import StableDiffusionPipeline
from PIL import Image
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. 加载模型

首次运行会自动下载模型（~4GB），之后从缓存加载。

In [ ]:
model_id = 'runwayml/stable-diffusion-v1-5'

pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    safety_checker=None,
)
pipe = pipe.to(device)
pipe.enable_attention_slicing()  # 节省显存
print('Model loaded!')

## 2. 文生图 (Text-to-Image)

In [ ]:
prompt = 'a beautiful sunset over the ocean, digital art, highly detailed'
negative = 'low quality, blurry, distorted'

generator = torch.Generator(device).manual_seed(42)
image = pipe(
    prompt=prompt,
    negative_prompt=negative,
    num_inference_steps=30,
    guidance_scale=7.5,
    generator=generator,
).images[0]

plt.imshow(image)
plt.title(f'Prompt: {prompt}')
plt.axis('off')
plt.show()

## 3. 批量生成对比

In [ ]:
prompts = [
    'a cat sitting on a couch, photorealistic',
    'a futuristic city at night, cyberpunk style',
    'a watercolor painting of a garden',
    'an astronaut riding a horse on mars',
]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for i, (ax, prompt) in enumerate(zip(axes, prompts)):
    generator = torch.Generator(device).manual_seed(i)
    img = pipe(prompt=prompt, num_inference_steps=25, generator=generator).images[0]
    ax.imshow(img)
    ax.set_title(prompt[:40] + '...', fontsize=9)
    ax.axis('off')
plt.suptitle('Different Prompts', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Guidance Scale 对比

CFG 值越高，越贴合 prompt，但可能过拟合。

In [ ]:
cfg_values = [1.0, 3.0, 7.5, 12.0, 20.0]
prompt = 'a red rose in a vase, detailed'

fig, axes = plt.subplots(1, len(cfg_values), figsize=(20, 4))
for ax, cfg in zip(axes, cfg_values):
    generator = torch.Generator(device).manual_seed(42)
    img = pipe(prompt=prompt, guidance_scale=cfg, num_inference_steps=30,
               generator=generator).images[0]
    ax.imshow(img)
    ax.set_title(f'CFG = {cfg}')
    ax.axis('off')
plt.suptitle('Guidance Scale Comparison', fontsize=14)
plt.tight_layout()
plt.show()

## 5. 图生图 (Image-to-Image)

In [ ]:
from diffusers import StableDiffusionImg2ImgPipeline

# 加载 img2img pipeline
pipe_img2img = StableDiffusionImg2ImgPipeline.from_pretrained(
    model_id, torch_dtype=torch.float16, safety_checker=None
).to(device)

# 使用上面生成的图作为输入
init_image = image.resize((512, 512))

strengths = [0.3, 0.5, 0.7, 0.9]
fig, axes = plt.subplots(1, len(strengths) + 1, figsize=(20, 4))
axes[0].imshow(init_image)
axes[0].set_title('Original')
axes[0].axis('off')

for ax, s in zip(axes[1:], strengths):
    generator = torch.Generator(device).manual_seed(42)
    img = pipe_img2img(
        prompt='a sunset painting, impressionist style',
        image=init_image,
        strength=s,
        generator=generator,
    ).images[0]
    ax.imshow(img)
    ax.set_title(f'Strength = {s}')
    ax.axis('off')
plt.suptitle('Image-to-Image: Strength Comparison', fontsize=14)
plt.tight_layout()
plt.show()

## 6. LoRA 推理

加载微调后的 LoRA 权重进行推理。

**注意**: 需要先运行 `sd_finetune/train_lora.py` 训练 LoRA。

In [ ]:
# 如果有 LoRA 权重，取消注释以下代码
# lora_path = '../sd_finetune/checkpoints/lora/final'
# pipe_lora = StableDiffusionPipeline.from_pretrained(
#     model_id, torch_dtype=torch.float16, safety_checker=None
# )
# pipe_lora.unet.load_adapter(lora_path)
# pipe_lora = pipe_lora.to(device)
#
# generator = torch.Generator(device).manual_seed(42)
# img = pipe_lora('a photo of sks dog', generator=generator).images[0]
# plt.imshow(img)
# plt.axis('off')
# plt.show()

## 总结

本 notebook 演示了完整 Stable Diffusion 的核心功能：
1. **文生图**: 从文字描述生成图像
2. **CFG 调节**: 控制 prompt 遵循程度
3. **图生图**: 基于原图进行风格迁移
4. **LoRA**: 加载微调权重生成特定风格

更多功能请参考 `sd_inference/` 目录下的脚本。